In [3]:
import pandas as pd
import numpy as np
import os
import torch
import seaborn as sns


from scipy.stats import pearsonr, spearmanr
from matplotlib import pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve, auc


from Bio import pairwise2
from Bio.Seq import Seq
from Bio.Align import substitution_matrices
from Bio import SeqIO

from random import sample

from utils_for_analysis import (
    calculate_ss_for_df_and_factors,
    load_df_all,
    load_df_with_budget,
    discretized_parameter_scale,
    xlabel_dict,
    ylabel_dict,
    title_fontsize,
    label_fontsize,
    tick_fontsize,
    legend_fontsize,
    original_parameter_scale,
    color_map,
    fix_ticks,
    get_labels,
    positions,
    num_muts_column_name,
    DATASET_PATHS
)


/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [4]:
datasets_and_activity = {
    "gcn4": {"path": "./data/gcn4/gcn4.csv", "activity_col": "activity"},
    "pard3": {"path": "./data/pard3/pard3.csv", "activity_col": "activity"},
    "pte": {"path": "./data/pte/pte.csv", "activity_col": "activity"},
    "nmt": {"path": "./data/nmt/nmt_full_seq.csv", "activity_col": "activity"},
    "lov": {"path": "./data/lov/lov.csv", "activity_col": "activity"},
    "aamyl": {"path": "./data/aamyl/aamyl.csv", "activity_col": "activity"},
    "gfp": {"path": "./data/gfp/gfp_dataset_10mut.csv", "activity_col": "inactive"},
    "trpb": {"path": "./data/trpb/trpb.csv", "activity_col": "activity"},
    "his": {"path": "./data/his/his.csv", "activity_col": "activity"}
}

In [7]:
all_results = []

for dataset, dataset_path in dict([("casp", DATASET_PATHS["casp"])]).items():
    scores = []

    df = pd.read_csv(dataset_path)
    pssm = pd.read_csv(f"./data/{dataset}//pssm_scores.csv")

    wt_df = df[df[num_muts_column_name[dataset]] == 0]
    si = np.where(df.columns == positions[dataset][0])[0][0]
    ei = np.where(df.columns == positions[dataset][1])[0][0]+1
    dataset_positions = np.array([int(c[1:]) for c in df.columns[si:ei].tolist()]) - 1  # PDB BASED

    wt_seq = wt_df.iloc[:, si:ei].values[0]
    col_pos_in_pssm = np.array([np.where(seq_aa == pssm.columns)[0][0] for seq_aa in wt_seq])
    wt_score = pssm.values[dataset_positions, col_pos_in_pssm]

    nmuts = df[num_muts_column_name[dataset]].to_numpy()
    nmuts_with_zeros = nmuts.copy()
    nmuts_with_zeros[nmuts_with_zeros == 0] = 1

    for idx, row in df.iterrows():
        mut_seq = row.iloc[si:ei].values
        col_pos_in_pssm = np.array([np.where(seq_aa == pssm.columns)[0][0] for seq_aa in mut_seq])
        mut_score = pssm.values[dataset_positions, col_pos_in_pssm]
        exp_score = np.exp(mut_score)
        raw_score = np.sum(mut_score)
        avg_score = np.sum(mut_score / nmuts_with_zeros[idx])
        raw_fitness = np.sum(mut_score - wt_score)
        avg_fitness = np.sum((mut_score - wt_score) / nmuts_with_zeros[idx])
        scores.append([raw_score, avg_score, raw_fitness, avg_fitness])
        if idx % 1000 == 0:
            print(f"Processed {idx} rows")

    labels = get_labels(dataset)
    df_scores = pd.DataFrame(scores, columns=["raw_score", "avg_score", "raw_fitness", "avg_fitness"])
    all_nmuts = np.unique(nmuts)

    for discrete in [True, False]:
        if discrete:
            scoring_func = lambda y_true, y_pred: roc_auc_score(y_true, y_pred)
            comparison_type = "aucroc"
            y_labels = (labels > np.median(labels)).astype(int)
        else:
            if dataset == "gfp":
                continue

            scoring_func = lambda y_true, y_pred: spearmanr(y_true, y_pred)[0]
            comparison_type = "spearman"
            y_labels = labels

        # All nmuts aggregated, use mutations=-1
        for col in df_scores.columns:
            try:
                result_value = scoring_func(y_labels, df_scores[col])
            except Exception:
                result_value = None
            all_results.append({
                "dataset": dataset,
                "value": result_value,
                "comparision_type": comparison_type,
                "mutations": -1,
                "score_type": col
            })

        # Per nmuts group (skip nmuts==0)
        for specific_nmuts in all_nmuts:
            if specific_nmuts == 0:
                continue
            mask = nmuts == specific_nmuts
            if not np.any(mask):
                continue
            for col in df_scores.columns:
                try:
                    value = scoring_func(y_labels[mask], df_scores[col][mask])
                except Exception:
                    value = None
                all_results.append({
                    "dataset": dataset,
                    "value": value,
                    "comparision_type": comparison_type,
                    "mutations": int(specific_nmuts),
                    "score_type": col
                })

final_score_df = pd.DataFrame(all_results, columns=["dataset", "value", "comparision_type", "mutations", "score_type"])




Processed 0 rows
Processed 1000 rows
Processed 2000 rows
Processed 3000 rows
Processed 4000 rows
Processed 5000 rows
Processed 6000 rows
Processed 7000 rows
Processed 8000 rows
Processed 9000 rows
Processed 10000 rows
Processed 11000 rows
Processed 12000 rows
Processed 13000 rows
Processed 14000 rows
Processed 15000 rows
Processed 16000 rows
Processed 17000 rows
Processed 18000 rows


/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages

In [8]:
final_score_df[final_score_df["mutations"] == -1]

,dataset,value,comparision_type,mutations,score_type
0,casp,0.716141,aucroc,-1,raw_score
1,casp,0.700737,aucroc,-1,avg_score
2,casp,0.716141,aucroc,-1,raw_fitness
3,casp,0.621804,aucroc,-1,avg_fitness
76,casp,0.452891,spearman,-1,raw_score
77,casp,0.427986,spearman,-1,avg_score
78,casp,0.452891,spearman,-1,raw_fitness
79,casp,0.238349,spearman,-1,avg_fitness


In [23]:
auc_roc = final_score_df["value"]
auc_roc[(auc_roc < 0.5) & (final_score_df["comparision_type"] == "spearman")] = 1 - auc_roc[(auc_roc < 0.5) & (final_score_df["comparision_type"] == "spearman")]
final_score_df[(final_score_df["mutations"] == -1) & 
               (final_score_df["comparision_type"] == "spearman")]




/tmp/ipykernel_2017628/2425314634.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  auc_roc[(auc_roc < 0.5) & (final_score_df["comparision_type"] == "spearman")] = 1 - auc_roc[(auc_roc < 0.5) & (final_score_df["comparision_type"] == "spearman")]


,dataset,value,comparision_type,mutations,score_type
20,trpb,0.646676,spearman,-1,raw_score
21,trpb,1.104852,spearman,-1,avg_score
22,trpb,0.646676,spearman,-1,raw_fitness
23,trpb,0.596338,spearman,-1,avg_fitness


In [71]:
df_scores = pd.DataFrame(scores, columns=["raw_score", "avg_score", "raw_fitness", "avg_fitness",
                                          "exp_raw_score", "exp_avg_score", "exp_raw_fitness", "exp_avg_fitness"])


all_nmuts = np.unique(nmuts)
discrete = True

if discrete and dataset != "gfp":
    labels = (labels > np.median(labels)).astype(int)


for col in df_scores.columns:
    print("\n\n########################")
    print(col)
    print("########################")
    if discrete:
        print(roc_auc_score(labels, df_scores[col]))
    else:
        print(spearmanr(labels, df_scores[col]))

    for specific_nmuts in all_nmuts:
        if specific_nmuts == 0:
            continue

        mask = nmuts == specific_nmuts
        if discrete:
            print(roc_auc_score(labels[mask], df_scores[col][mask]))
        else:
            print(spearmanr(labels[mask], df_scores[col][mask]))




########################
raw_score
########################
0.7098926571840922
nan
0.779
0.7266974791141179
0.6829733202236686
0.6675382357350291
0.6613541888560908
0.6591803509741891
0.6650216497683817
0.6742520851294282
0.6873440269425931


########################
avg_score
########################
0.7098495347619789
nan
0.779
0.7271421697989415
0.6829733202236686
0.6659914081551306
0.6618264220673611
0.6597817778586978
0.6650216497683817
0.67427857264735
0.6870239978175119


########################
raw_fitness
########################
0.7098926571840922
nan
0.779
0.7266974791141179
0.6829733202236686
0.6675382357350291
0.6613541888560908
0.6591803509741891
0.6650216497683817
0.6742520851294282
0.6873440269425931


########################
avg_fitness
########################
0.6612212835087266
nan
0.779
0.7258737078455101
0.6829733202236686
0.66748785074222
0.6612032500532545
0.6593011365203127
0.6650216497683817
0.67383682849481
0.6877517215037157


########################
exp

/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages

0.612874549043389
nan
0.638
0.624462361672037
0.5830021986838058
0.5638518140008524
0.559240251023672
0.5454662993458957
0.5457941791585191
0.546070161613393
0.5468297239833229


########################
exp_avg_score
########################
0.6343905258923789
nan
0.638
0.62449881172817
0.5830021986838058
0.5638506934120502
0.5592375882927753
0.5454642586053912
0.5457941791585191
0.5460719430661369
0.5468333755894252


########################
exp_raw_fitness
########################
0.6128745448977995
nan
0.638
0.6244550716608103
0.5830010676736176
0.5638516479876965
0.5592401918518743
0.5454662930859555
0.5457941998128323
0.5460701494116618
0.5468295895353811


########################
exp_avg_fitness
########################
0.6342557599480626
nan
0.638
0.6244113315934506
0.5830010676736176
0.5638482032147119
0.5592410794288399
0.5454668502206332
0.5457941998128323
0.5460715135652013
0.5468343973937836


/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/labs/fleishman/itayta/.conda/envs/esm_env/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [102]:
nmuts

0

array([ 8,  8,  8,  7,  7,  8,  7,  8,  8,  8,  8,  7,  8,  8, -1,  8,  8,
        7,  8,  8,  8,  8])

In [48]:
pssm.columns

Index(['position', 'A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K',
       'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V'],
      dtype='object')